# 04 — Three-Layer Molecular Graph Neural Network (GNN) Encoder
**Course:** 22AIE401 — Reinforcement Learning  
**Project:** Reinforcement Learning for Drug-Likeness Optimization of Small Molecules (Post-Mid-Review Continuation)  
**Dataset:** ZINC-250K  

---

## 1. Overview & Architectural Specifications
This notebook demonstrates and validates the **Three-Layer Molecular Graph Neural Network (GNN) Encoder** designed to encode small molecules into compact, permutation-invariant continuous latent representations $z \in \mathbb{R}^{128}$ for downstream PPO Actor-Critic policy evaluation.

### Key Architectural Components:
1. **Node Featurization:** $x_v \in \mathbb{R}^{21}$ (Atomic element one-hot across 11 organic elements + other, degree one-hot 0-5, formal charge, implicit/explicit H count, and aromaticity).
2. **Edge Featurization:** $e_{uv} \in \mathbb{R}^4$ (Bond type one-hot: Single, Double, Triple, Aromatic). Bidirectional graph conversion.
3. **Message Passing:** 3-layer `GraphConv` with edge-attribute conditioning:
   $$h_v^{(l)} = W_1^{(l)} h_v^{(l-1)} + \sum_{u \in \mathcal{N}(v)} W_2^{(l)} h_u^{(l-1)} \cdot e_{u,v}$$
4. **Dual Global Pooling:** Concatenation of Global Mean and Global Max pooling across all atom representations:
   $$h_{\text{graph}} = [\text{mean\_pool}(H^{(3)}), \text{max\_pool}(H^{(3)})] \in \mathbb{R}^{128}$$
5. **Scalar State Conditioning:** Integration of dynamic MDP environment features ($6$ dimensions: Current QED, SAS/10, Tanimoto similarity to starting molecule $M_0$, normalized step $t/T_{\max}$, target difference, constraint slack).
6. **Latent Representation:** Fusion MLP producing final state vector $z \in \mathbb{R}^{128}$.


In [1]:
import os, sys
import torch
from rdkit import Chem
from rdkit.Chem import Descriptors
import pandas as pd
import numpy as np

# Verify modules
from antigravity_workspace.implementation.gnn_encoder import (
    MolecularGNNEncoder,
    mol_to_pyg_data,
    get_node_features,
    get_edge_features
)

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print("GNN Encoder modules loaded successfully.")

PyTorch version: 2.9.0+cpu
Device: cpu
GNN Encoder modules loaded successfully.


## 2. Molecular Graph Extraction
We load a representative drug-like molecule from the ZINC-250K training split and extract the PyTorch Geometric graph data representation.

In [2]:
# Load a sample molecule from training split
train_path = os.path.join(WORKSPACE_DIR, 'data', 'splits', 'train_molecules.csv')
train_df = pd.read_csv(train_path)
sample_row = train_df.iloc[0]
smiles = sample_row['smiles']
mol = Chem.MolFromSmiles(smiles)

print(f"Sample Molecule SMILES: {smiles}")
print(f"Formula: {Chem.rdMolDescriptors.CalcMolFormula(mol)}")
print(f"Molecular Weight: {Descriptors.MolWt(mol):.2f} Da")
print(f"Ground Truth QED: {sample_row['qed']:.4f}")

# Convert to PyG Graph
pyg_data = mol_to_pyg_data(mol)
print("\n--- PyTorch Geometric Data Object ---")
print(pyg_data)
print(f"Number of Atoms (Nodes): {pyg_data.num_nodes}")
print(f"Number of Directed Edges: {pyg_data.num_edges} (corresponding to {pyg_data.num_edges // 2} chemical bonds)")
print(f"Node Feature Matrix Shape (x): {pyg_data.x.shape} -> Expected [N, 21]")
print(f"Edge Index Shape: {pyg_data.edge_index.shape} -> Expected [2, 2E]")
print(f"Edge Attribute Shape: {pyg_data.edge_attr.shape} -> Expected [2E, 4]")

Sample Molecule SMILES: CCc1nn(C)c(OC)c1CNC(=O)Cc1ccc(Cl)cc1
Formula: C16H20ClN3O2
Molecular Weight: 321.81 Da
Ground Truth QED: 0.8880

--- PyTorch Geometric Data Object ---
Data(x=[22, 21], edge_index=[2, 46], edge_attr=[46, 4], scalars=[1, 6], num_atoms=22)
Number of Atoms (Nodes): 22
Number of Directed Edges: 46 (corresponding to 23 chemical bonds)
Node Feature Matrix Shape (x): torch.Size([22, 21]) -> Expected [N, 21]
Edge Index Shape: torch.Size([2, 46]) -> Expected [2, 2E]
Edge Attribute Shape: torch.Size([46, 4]) -> Expected [2E, 4]


## 3. Node and Edge Feature Matrix Inspection
Here we inspect the explicit atom feature encodings for the first 5 atoms in the molecule, verifying element one-hot encoding, degree, and valence features.

In [3]:
# Display first 5 atom feature vectors
print("First 5 Atom Feature Vectors [x[:5]] shape:", pyg_data.x[:5].shape)
print("Node features sample (Atom 0):", pyg_data.x[0].numpy())
print("\nFirst 4 Directed Edge Attributes [edge_attr[:4]]:")
edge_features_df = pd.DataFrame(pyg_data.edge_attr[:4].numpy(), columns=['Single', 'Double', 'Triple', 'Aromatic'])
print(edge_features_df.to_string())

First 5 Atom Feature Vectors [x[:5]] shape: torch.Size([5, 21])
Node features sample (Atom 0): [1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]

First 4 Directed Edge Attributes [edge_attr[:4]]:
   Single  Double  Triple  Aromatic
0     1.0     0.0     0.0       0.0
1     1.0     0.0     0.0       0.0
2     1.0     0.0     0.0       0.0
3     1.0     0.0     0.0       0.0


## 4. GNN Encoder Instantiation
We instantiate the `MolecularGNNEncoder` model with the specified hyperparameters and verify its layer composition and parameter count.

In [4]:
# Initialize 3-Layer GNN
gnn = MolecularGNNEncoder(
    node_dim=21,
    edge_dim=4,
    gnn_hidden=64,
    scalar_dim=6,
    fused_dim=128
)

total_params = sum(p.numel() for p in gnn.parameters())
trainable_params = sum(p.numel() for p in gnn.parameters() if p.requires_grad)

print(gnn)
print(f"\nTotal Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

MolecularGNNEncoder(
  (conv1): GraphConv(21, 64)
  (conv2): GraphConv(64, 64)
  (conv3): GraphConv(64, 64)
  (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (bn3): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (graph_proj): Linear(in_features=128, out_features=64, bias=True)
  (fusion_mlp): Sequential(
    (0): Linear(in_features=70, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
  )
)

Total Parameters: 53,504
Trainable Parameters: 53,504


## 5. Forward Pass Verification (Single Molecule)
We perform a complete forward pass with simulated environment scalar state features:
$s = [\text{QED}=0.55, \text{SAS}/10=0.32, \text{Sim}=0.85, t/T=0.30, \Delta_{\text{target}}=0.25, \text{slack}=0.45]$.  
We verify the output tensor shape is strictly `[1, 128]`.

In [5]:
gnn.eval()
with torch.no_grad():
    # Construct scalar state vector [1, 6]
    scalar_state = torch.tensor([[0.55, 0.32, 0.85, 0.30, 0.25, 0.45]], dtype=torch.float32)
    
    # Run forward pass
    embedding = gnn(pyg_data, scalar_state)

print(f"Output Embedding Shape: {embedding.shape}")
print(f"Embedding Mean: {embedding.mean().item():.4f}")
print(f"Embedding Std: {embedding.std().item():.4f}")
print(f"Contains NaNs: {torch.isnan(embedding).any().item()}")
print(f"Contains Infs: {torch.isinf(embedding).any().item()}")
assert embedding.shape == (1, 128), "Error: Output embedding must be [1, 128]"
print("PASS: Forward pass single molecule test passed successfully.")

Output Embedding Shape: torch.Size([1, 128])
Embedding Mean: 0.0411
Embedding Std: 0.0594
Contains NaNs: False
Contains Infs: False
PASS: Forward pass single molecule test passed successfully.


## 6. Batched Graph Processing Verification
In PPO experience collection and mini-batch policy updates, molecules are batched into a single disjoint graph block. We test PyTorch Geometric `Batch` processing across a cohort of 8 distinct molecules.

In [6]:
from torch_geometric.data import Batch

# Sample 8 diverse molecules
batch_smiles = train_df['smiles'].iloc[:8].tolist()
batch_pyg_list = [mol_to_pyg_data(Chem.MolFromSmiles(s)) for s in batch_smiles]
batched_data = Batch.from_data_list(batch_pyg_list)

# Batched scalar states [8, 6]
batch_scalars = torch.rand(8, 6, dtype=torch.float32)

with torch.no_grad():
    batched_embeddings = gnn(batched_data, batch_scalars)

print(f"Batched Data Total Nodes: {batched_data.num_nodes}")
print(f"Batched Data Total Edges: {batched_data.num_edges}")
print(f"Batched Output Embedding Shape: {batched_embeddings.shape}")
assert batched_embeddings.shape == (8, 128), "Error: Batched output must be [8, 128]"
print("PASS: Batched processing test passed successfully.")

Batched Data Total Nodes: 202
Batched Data Total Edges: 442
Batched Output Embedding Shape: torch.Size([8, 128])
PASS: Batched processing test passed successfully.


## 7. Latent Representation Sanity & Gradient Flow Checks
To guarantee that the GNN is not collapsing representations to a constant and supports stable backpropagation:
1. We compute pairwise cosine similarity across embeddings of distinct molecules.
2. We verify smooth gradient propagation from the embedding back to all GNN weights.

In [7]:
# 1. Cosine similarity matrix across 4 diverse molecules
sub_embeds = batched_embeddings[:4]
norm_embeds = sub_embeds / sub_embeds.norm(dim=1, keepdim=True)
cosine_sim_matrix = torch.mm(norm_embeds, norm_embeds.t()).numpy()

print("Cosine Similarity Matrix across 4 distinct molecules:")
cos_df = pd.DataFrame(cosine_sim_matrix, index=[f"Mol_{i}" for i in range(4)], columns=[f"Mol_{i}" for i in range(4)])
print(cos_df.round(4).to_string())

# 2. Gradient flow test
gnn.train()
gnn.zero_grad()
dummy_scalar = torch.rand(1, 6, requires_grad=True)
test_out = gnn(pyg_data, dummy_scalar)
loss = test_out.sum()
loss.backward()

grad_norms = [p.grad.norm().item() for p in gnn.parameters() if p.grad is not None]
print(f"\nGradient Check:")
print(f"Parameters with gradients: {len(grad_norms)} / {len(list(gnn.parameters()))}")
print(f"Mean Gradient Norm: {np.mean(grad_norms):.6f}")
print(f"Max Gradient Norm: {np.max(grad_norms):.6f}")
assert len(grad_norms) > 0 and all(g > 0 for g in grad_norms), "Error: Zero or missing gradients detected!"
print("PASS: Representation sanity and gradient flow tests verified.")

Cosine Similarity Matrix across 4 distinct molecules:
        Mol_0   Mol_1   Mol_2   Mol_3
Mol_0  1.0000  0.9627  0.9470  0.9482
Mol_1  0.9627  1.0000  0.9688  0.9600
Mol_2  0.9470  0.9688  1.0000  0.9881
Mol_3  0.9482  0.9600  0.9881  1.0000

Gradient Check:
Parameters with gradients: 21 / 21
Mean Gradient Norm: 7.067628
Max Gradient Norm: 31.342983
PASS: Representation sanity and gradient flow tests verified.
